In [10]:
import nibabel as nib
import numpy as np
import os
from scipy.optimize import lsq_linear
import sys
import pandas as pd

sub_name = "sub-03"
root_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
data_dir = f"{root_dir}/data/{sub_name}"
sessions = [f"ses-{i:02d}" for i in [1, 2, 3, 4, 5, 6, 7, 8, 10]]
base_session = "ses-01"

In [11]:
def load_nii(path):
    img = nib.load(path)
    data = img.get_fdata()    
    return data, img.affine

In [ ]:
def opt_core_shim_py(mapB0, mapShim, param):
    """
    Python version of opt_core_shim using scipy.optimize.lsq_linear.
    
    Arguments:
        mapB0: B0 map of the brain, shape (nx, ny, nz)
        mapShim: shim profiles, shape (num_coils, nx, ny, nz)
        param: dict containing 'shim_mode', 'coilN', 'shimCLL', 'shimCUL'
        
    Returns:
        shimmed: Corrected B0 map, same shape as mapB0
        coef: Optimization coefficients (currents)
    """
    mapB0_bu = mapB0.copy()
    mapShim_bu = mapShim.copy()
    
    num_coils = int(param['coilN'])
    lb = np.array(param['shimCLL']).flatten()
    ub = np.array(param['shimCUL']).flatten()

    if param['shim_mode'] == 'global':
        mapShim_flat = mapShim_bu.reshape(num_coils, -1)
        mapB0_flat = mapB0_bu.reshape(-1)
        
        mask = ~np.isnan(mapB0_flat)
        A = mapShim_flat[:, mask].T 
        b = -mapB0_flat[mask]       

        res = lsq_linear(A, b, bounds=(lb, ub), method='trf')
        coef = res.x

        shimmed = np.tensordot(coef, mapShim_bu, axes=(0, 0)) + mapB0_bu
        
    else:
        mode = param['shim_mode']
        if mode == 'slice_wise_tra':   # 轴向: 逐 Z 层 (轴 2)
            permute_order = (0, 1, 2)
        elif mode == 'slice_wise_cor': # 冠状: 逐 Y 层 (轴 1)
            permute_order = (0, 2, 1)
        elif mode == 'slice_wise_sag': # 矢状: 逐 X 层 (轴 0)
            permute_order = (1, 2, 0)
        else:
            raise ValueError("Invalid shim mode")
        mB0 = mapB0_bu.transpose(permute_order)
        mShim = mapShim_bu.transpose((0,) + tuple(a + 1 for a in permute_order))
        
        nx_p, ny_p, nz_p = mB0.shape
        shimmed_p = np.zeros_like(mB0)
        coef = np.zeros((num_coils, nz_p))
        
        for i in range(nz_p):
            mB0_slice = mB0[:, :, i].reshape(-1)
            mShim_slice = mShim[:, :, :, i].reshape(num_coils, -1)
            
            mask = ~np.isnan(mB0_slice)
            if np.sum(mask) == 0:
                shimmed_p[:, :, i] = mB0[:, :, i] 
                continue
                
            A = mShim_slice[:, mask].T
            b = -mB0_slice[mask]
            
            res = lsq_linear(A, b, bounds=(lb, ub), method='trf')
            coef_i = res.x
            coef[:, i] = coef_i
            
            shim_slice_res = np.tensordot(coef_i, mShim[:, :, :, i], axes=(0, 0))
            shimmed_p[:, :, i] = shim_slice_res + mB0[:, :, i]
            
        inv_permute = np.argsort(permute_order)
        shimmed = shimmed_p.transpose(inv_permute)

    return shimmed, coef

In [ ]:
def opt_core_shim_py(mapB0, mapShim, param):
    """
    Python version of opt_core_shim using scipy.optimize.lsq_linear.
    
    Arguments:
        mapB0: B0 map of the brain, shape (nx, ny, nz)
        mapShim: shim profiles, shape (num_coils, nx, ny, nz)
        param: dict containing 'shim_mode', 'coilN', 'shimCLL', 'shimCUL'
        
    Returns:
        shimmed: Corrected B0 map, same shape as mapB0
        coef: Optimization coefficients (currents)
    """
    mapB0_bu = mapB0.copy()
    mapShim_bu = mapShim.copy()
    
    num_coils = int(param['coilN'])
    lb = np.array(param['shimCLL']).flatten()
    ub = np.array(param['shimCUL']).flatten()

    if param['shim_mode'] == 'global':
        mapShim_flat = mapShim_bu.reshape(num_coils, -1)
        mapB0_flat = mapB0_bu.reshape(-1)
        
        mask = ~np.isnan(mapB0_flat)
        A = mapShim_flat[:, mask].T 
        b = -mapB0_flat[mask]       

        res = lsq_linear(A, b, bounds=(lb, ub), method='trf')
        coef = res.x

        shimmed = np.tensordot(coef, mapShim_bu, axes=(0, 0)) + mapB0_bu
        
    else:
        mode = param['shim_mode']
        if mode == 'slice_wise_tra':   # 轴向: 逐 Z 层 (轴 2)
            permute_order = (0, 1, 2)
        elif mode == 'slice_wise_cor': # 冠状: 逐 Y 层 (轴 1)
            permute_order = (0, 2, 1)
        elif mode == 'slice_wise_sag': # 矢状: 逐 X 层 (轴 0)
            permute_order = (1, 2, 0)
        else:
            raise ValueError("Invalid shim mode")
        mB0 = mapB0_bu.transpose(permute_order)
        mShim = mapShim_bu.transpose((0,) + tuple(a + 1 for a in permute_order))
        
        nx_p, ny_p, nz_p = mB0.shape
        shimmed_p = np.zeros_like(mB0)
        coef = np.zeros((num_coils, nz_p))
        
        for i in range(nz_p):
            mB0_slice = mB0[:, :, i].reshape(-1)
            mShim_slice = mShim[:, :, :, i].reshape(num_coils, -1)
            
            mask = ~np.isnan(mB0_slice)
            if np.sum(mask) == 0:
                shimmed_p[:, :, i] = mB0[:, :, i] 
                continue
                
            A = mShim_slice[:, mask].T
            b = -mB0_slice[mask]
            
            res = lsq_linear(A, b, bounds=(lb, ub), method='trf')
            coef_i = res.x
            coef[:, i] = coef_i
            
            shim_slice_res = np.tensordot(coef_i, mShim[:, :, :, i], axes=(0, 0))
            shimmed_p[:, :, i] = shim_slice_res + mB0[:, :, i]
            
        inv_permute = np.argsort(permute_order)
        shimmed = shimmed_p.transpose(inv_permute)

    return shimmed, coef

In [13]:
is_jupyter = any('jupyter' in arg.lower() for arg in sys.argv) or 'ipykernel' in sys.modules

if not is_jupyter:
    data_dir = sys.argv[1] 
    subname = sys.argv[2] 
    sessions = sys.argv[3].split()
    base_session = sys.argv[4]

coil_sessions = sessions[1:]  # 线圈 1 到 n

base_path = os.path.join(data_dir, base_session, "B0_map_Hz_to_master_masked.nii.gz")
base_data, affine = load_nii(base_path)

# 读取线圈场并计算基函数 (Coil - Baseline)
b1_z_list = []
for sess in coil_sessions:
    p = os.path.join(data_dir, sess, "B0_map_Hz_to_master_masked.nii.gz")
    coil_data, _ = load_nii(p)
    b1_z_list.append(coil_data - base_data)

b1_z = np.array(b1_z_list)  # 维度: [num_coils, nx, ny, nz]
nx, ny, nz = base_data.shape
num_coils = b1_z.shape[0]

print(f"数据读取完毕。共有 {num_coils} 个线圈，空间维度为 {nx}x{ny}x{nz}")

数据读取完毕。共有 8 个线圈，空间维度为 104x104x72


In [ ]:
shimCL = 2.0  # 电流上限 (A)

param = {
    'shim_mode': 'slice_wise_tra', 
    'coilN': num_coils,
    'shimCUL': np.full(num_coils, shimCL), 
    'shimCLL': np.full(num_coils, -shimCL),
    'shim_algo_inner': 'lsqlin' 
}
# 确保非脑组织区域为 NaN
base_data_input = np.where((base_data == 0) | (~np.isfinite(base_data)), np.nan, base_data)

shimmed_brain, currents = opt_core_shim_py(base_data_input, b1_z, param)
file_path = os.path.join(data_dir, "shimming_currents.csv")
pd.DataFrame(currents).to_csv(file_path,  index=False)
print(currents)

[[ 2.50000000e-01  2.50000000e-01  2.35879148e-16 -7.62531462e-16
  -1.90350769e-15 -2.23159297e-15 -2.87047254e-03  2.79185540e-03
  -1.63745293e-02 -6.27341219e-02 -6.81611715e-02 -1.69630828e-01
  -2.07186577e-01 -1.71612010e-01 -1.07379536e-01 -8.48561325e-03
   3.53272682e-02  5.97746719e-02  6.15089140e-02  4.73719465e-02
  -1.31741212e-02 -1.10655544e-01 -1.62783998e-01 -1.86737352e-01
  -1.94569180e-01 -1.90469524e-01 -1.92053874e-01 -1.80463188e-01
  -1.77614591e-01 -1.78781577e-01 -1.68392155e-01 -1.62986925e-01
  -1.64371721e-01 -1.48695929e-01 -1.33115320e-01 -9.31936874e-02
  -3.63380518e-02  2.68143630e-02  8.55271388e-02  1.36083893e-01
   1.77360937e-01  2.01432776e-01  1.62595036e-01  2.18409004e-02
  -1.85931143e-01 -2.60148147e-01 -2.44157947e-01 -1.90070420e-01
  -1.32578345e-01 -9.31743856e-02 -1.37335719e-01 -1.73855343e-01
  -2.45767000e-01 -2.94212920e-01 -2.90972931e-01 -2.72235072e-01
  -2.69198257e-01 -1.52416832e-01 -8.36408285e-02 -1.93077161e-01
  -2.29546